# Praktikum 5 — Pipeline Data Berita: dari Ingestion hingga MapReduce

**Mata kuliah:** Praktikum Big Data (DIF61142)  
**Nama:** Alfaris Aulia Rahman  
**NIM:** 2411533006


## G. Persiapan Praktikum

Buka Google Colab, buat notebook baru, dan pilih runtime CPU. Pasang pustaka berikut. Sel-sel saling memakai variabel, sehingga harus dijalankan berurutan.


In [2]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


## H-1. Peta Pipeline dan Permainan Kartu Kantin

| Tahap | Hasil | Langkah |
|---|---|---|
| Ingestion | `lapisan_mentah/` (JSONL) | H-3, H-4, H-5 |
| Quality dan cleaning | DataFrame bersih | H-6 |
| Storage | `lapisan_terkurasi/` (Parquet berpartisi) | H-6 |
| Query dan modeling | Tabel dan dokumen | H-7 |
| Processing (MapReduce) | Frekuensi kata | H-8–H-12 |
| Serving | `siap_pakai/` (Parquet dan database) | H-13 |

**Permainan kartu:** bagi kelas menjadi tiga kasir dan kelompok reducer. Map membuat pasangan menu–jumlah, shuffle mengelompokkan menu yang sama, reduce menjumlahkan porsinya. Ulangi dengan combiner di setiap kasir.

| Kasir A | Kasir B | Kasir C |
|---|---|---|
| nasi goreng, 1 | nasi goreng, 2 | mie ayam, 2 |
| es teh, 2 | es teh, 1 | nasi goreng, 1 |
| mie ayam, 1 | es teh, 3 | es teh, 1 |
| es teh, 1 | mie ayam, 1 | nasi goreng, 1 |

**Hasil:** nasi goreng 5, es teh 8, mie ayam 4; total 17 porsi. Tanpa combiner berpindah 12 kartu, dengan combiner 9 kartu. Pengurangan 3 kartu atau 25%. Hasil ini dicocokkan dengan kode di H-8.


## H-2. Persiapan: Unduh Data dan Jalankan Portal Lokal

Unduh CSV historis dan periksa ukuran, jumlah baris, dan kolom. Jika akses sumber gagal, unggah `detik_news_title.csv` ke direktori kerja Colab.


In [3]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")
if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas :", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom  :", df_sumber.shape)
print("Kolom         :", list(df_sumber.columns))


Ukuran berkas : 16.1 MB
Baris, kolom  : (91017, 4)
Kolom         : ['date', 'url', 'title', 'category']


### H-2. Menyalakan portal lokal

Sel kedua menyalakan portal berita mini di komputer kita. Kita tidak perlu memahami isi sel kedua, cukup tahu bahwa ia melayani halaman `/berita?hal=N` berisi 100 judul per halaman, serta berkas `robots.txt`.


In [4]:
import http.server, urllib.parse, html

HALAMAN_UKURAN = 100          # judul per halaman

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):          # matikan log
            pass
        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'

                        '</article>')
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)


Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


## H-3. Crawling Portal Lokal: Latihan Aman

Baca `robots.txt`: halaman berita boleh diambil, sedangkan `/arsip-privat/` dilarang.


In [5]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")


/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


### H-3. Parser HTML dan crawler

Bagian kunci: judul dan URL berasal dari `h2.judul a`, kategori dari `span.kategori`, tanggal ISO dari atribut `datetime` pada `time`. Crawler memeriksa robots, memberi jeda, dan mencoba ulang maksimal tiga kali. Uji pada tiga halaman pertama.


In [6]:
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({

            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)


Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


## H-4. Crawling Berita Sungguhan dari Sitemap detik.com

### Langkah 1. Tulis parser sitemap berita

Baca XML dengan BeautifulSoup dan parser `xml`. Ambil `title`, `publication_date`, kategori dari subdomain, serta ID dari pola `/d-<angka>/`. Entri tanpa URL atau judul dilewati.


In [7]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": urlparse(url).netloc.split(".")[0],   # news, inet, sport, ...
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })
    return hasil


### H-4. Langkah 2. Uji parser tanpa internet

Gunakan tiga entri contoh XML dari modul. ID yang diharapkan: 8729282, 8729280, 8729264; semuanya berkategori news dan bertanggal 2026-10-29. Masukan string dan bytes harus menghasilkan data yang sama. Contoh ini hanya untuk pengujian parser, bukan pengganti crawl langsung.


In [8]:
# Contoh XML: bentuk disalin dari sitemap berita asli (tiga entri hasil uji dosen, 29 Okt 2026).
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>

<news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>

    <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
    </news:news>
  </url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:", parse_sitemap_news(CONTOH_XML.encode("utf-8")) == parse_sitemap_news(CONTOH_XML))


{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True


### H-4. Langkah 3 — fungsi pengambilan yang sopan

Fungsi ini sudah siap pakai. Ada enam pengaman: robots.txt dibaca per host, hanya tiga sitemap yang diambil (`SITEMAP_DIPILIH`), maksimum tujuh judul per sitemap dan 20 judul total, jeda 2 detik antar-request, User-Agent yang menyebut identitas Anda, dan berhenti seketika bila server membalas 403 atau 429. **Ganti `GANTI_DENGAN_EMAIL_ANDA` dengan email Anda.**

In [9]:
import urllib.robotparser

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: alfarisauliarahman@gmail.com)"
DELAY = 2.0                 # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7        # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20            # batas seluruh crawling
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}
def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)


def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content            # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]


### H-4. Langkah 4 — jalankan dan periksa

Hasil disimpan di folder `contoh_crawl/`, terpisah dari `lapisan_mentah/` agar angka acuan pada langkah berikutnya tidak berubah.

In [10]:
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul   :", len(cek))
    print("Judul kosong      :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat      :", cek["url"].duplicated().sum())
    print("Tanggal kosong    :", cek["tanggal"].isna().sum())
    print("Kategori          :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))


Judul terkumpul   : 20
Judul kosong      : 0
URL duplikat      : 0
Tanggal kosong    : 0
Kategori          : {'news': 7, 'inet': 7, 'sport': 6}
                                                                            judul kategori    tanggal
                                     Polri Terbitkan SE Larangan Personel Ngevape     news 2026-11-01
            Sopir Truk Molen Picu Kecelakaan Tewaskan 6 Orang di Bekasi Ditangkap     news 2026-11-01
IPW Dukung Polisi Usut Pembakaran Warung Mie di Sukoharjo, Ingatkan Akuntabilitas     news 2026-11-01


## H-5. Channel ala Flume ke lapisan_mentah

Crawl seluruh 911 halaman **portal lokal**, bukan 91 ribu request ke detik. Source memasukkan event ke `queue.Queue(maxsize=2000)`, sink menulis batch 5.000 event ke JSONL. Channel penuh menahan source (*backpressure*). Pastikan event masuk sama dengan event keluar.


In [11]:
RAW = Path("lapisan_mentah"); RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)                       # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0
    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []
    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev); statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source); t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
waktu = time.perf_counter() - t0
print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink  :", f"{statistik['keluar']:,}")
print("File batch          :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3],
"...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)

print("Total ukuran mentah :", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total   :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")


Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 7.6 detik untuk 911 halaman


## H-6. Quality, Cleaning, dan Parquet Berpartisi

### H-6.1. Pemeriksaan kualitas

Bagian kunci: judul 3–30 kata, kategori dikenal, dan URL memuat pola ID artikel. Periksa juga judul kosong, tanggal tidak valid, dan semua salinan judul duplikat.


In [12]:
mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))],
                   ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong"      : d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata"         : jml_kata.between(3, 30),
        "kategori dikenal"        : d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid"           : pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid"       : d["url"].fillna("").str.contains(r"/d-\d+", regex=True),
        "judul tidak duplikat"    : ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>6,}  ({(~lulus).mean()*100:5.2f}%)")


Baris mentah: 91,017
judul tidak kosong       gagal:      0  ( 0.00%)
judul 3-30 kata          gagal:     78  ( 0.09%)
kategori dikenal         gagal:      0  ( 0.00%)
tanggal valid            gagal:      0  ( 0.00%)
url artikel valid        gagal:  5,640  ( 6.20%)
judul tidak duplikat     gagal:    235  ( 0.26%)


### H-6.2. Pembersihan data

Rapikan spasi, simpan judul duplikat pertama, pertahankan judul 3–30 kata, konversi tanggal secara eksplisit, dan buat `bulan` serta `id`. URL rusak dikosongkan, sedangkan judulnya tetap dipakai agar kategori inet tidak hilang dari analisis teks.


In [13]:
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")   # kosong bila url rusak
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True))
print("Baris setelah dibersihkan:", f"{len(bersih):,}", "| dibuang:", f"{len(mentah) - len(bersih):,}")


Baris setelah dibersihkan: 90,818 | dibuang: 199


### H-6.3. Penyimpanan Parquet berpartisi

Bandingkan partisi kategori, bulan, serta kategori + bulan. Simpan versi final berdasarkan kategori dan demonstrasikan partition pruning untuk finance.


In [14]:
import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

# BAGIAN KUNCI: bandingkan dua pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori = finance")


partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori = finance


## H-7. Query DuckDB dan Bentuk Dokumen

Baca Parquet dengan hive partitioning, hitung judul per kategori dan judul yang memuat `corona` per bulan. Ubah satu baris menjadi dokumen berisi daftar token.


In [15]:
import duckdb
con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")

print(con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df().to_string(index=False))
print()
print(con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan""").df().to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))


kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433



  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


## H-8. Mini MapReduce: Kasus Kantin

Map menghasilkan pasangan menu–jumlah, shuffle mengumpulkan nilai berdasarkan menu, dan reduce menjumlahkannya. Bandingkan jumlah pasangan sebelum dan sesudah combiner dengan H-1.


In [16]:
kasir = [
    ["nasi goreng,1", "es teh,2", "mie ayam,1", "es teh,1"],      # kasir A
    ["nasi goreng,2", "es teh,1", "es teh,3", "mie ayam,1"],      # kasir B
    ["mie ayam,2", "nasi goreng,1", "es teh,1", "nasi goreng,1"], # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner:", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))


map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner: 12
Pasangan dikirim dengan combiner: 9


## H-9. Word Count Terdistribusi: Partisi sebagai Input Split

Satu berkas Parquet menjadi satu input split, sehingga terdapat 9 tugas mapper. Ukuran split tidak seimbang. Hitung semua kata, tampilkan hasil tanpa stopword, lalu cek kesetaraan dengan Counter biasa.


In [17]:
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f"  {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)
hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")
print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f"  {k:<10} {n:>7,}")
top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f"  {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)


Jumlah input split: 9
  kategori=finance 14,141 judul
  kategori=food     4,314 judul
  kategori=health   4,914 judul
  kategori=hot     16,290 judul
  kategori=inet     5,577 judul
  kategori=news    32,321 judul
  kategori=oto      4,375 judul
  kategori=sport    2,433 judul
  kategori=travel   6,453 judul



Pasangan dari semua mapper:

 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
  di          27,390
  corona      17,239
  ini         10,124
  yang         8,668
  dan          6,785
  ke           5,592
  tak          5,096
  dari         4,697
  jadi         4,486
  untuk        3,723



10 kata teratas (tanpa stopword):
  corona      17,239
  rp           3,154
  virus        3,075
  kasus        2,743
  polisi       2,732
  positif      2,682
  indonesia    2,504
  warga        2,495
  rumah        2,465
  jokowi       2,317



Sama dengan Counter biasa: True


## H-10. Combiner dan Jumlah Data yang Berpindah

Pertanyaan: apakah menjalankan mapper di beberapa proses membuat pekerjaan lebih cepat, dan apa pengaruh combiner? Sel berikut mengukur waktu (median dari tiga kali jalan) dan menghitung pasangan yang dikirim ke reducer. Angka pasangan stabil; waktu bervariasi antar-komputer dan antar-kali-jalan.

In [18]:
from multiprocessing import Pool

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")

for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h):>20,}")
    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)


CPU tersedia: 10
skenario                     waktu (s)    pasangan dikirim


1 proses, tanpa combiner          0.116              825,601


1 proses, dengan combiner         0.137               74,009


2 proses, tanpa combiner          0.207              825,601


2 proses, dengan combiner         0.118               74,009

Hasil dengan combiner sama persis: True


## H-11. Data Skew dan Beban Reducer

Gunakan CRC32 agar penetapan key ke reducer stabil. Beban dihitung sebagai jumlah pasangan yang diterima, bukan jumlah frekuensi yang direpresentasikan pasangan. Bandingkan tanpa/dengan combiner, kemudian 4, 16, dan 64 reducer.


In [19]:
R = 4        # jumlah reducer

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"{nama:<22}", " ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))


print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f"  {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f"  {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")


tanpa combiner         R0=240,836 (29.2%) R1=170,066 (20.6%) R2=190,727 (23.1%) R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%) R1= 18,267 (24.7%) R2= 18,530 (25.0%) R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
    4 reducer: terberat 29.17% | bagian adil 25.00%


   16 reducer: terberat  8.82% | bagian adil  6.25%
   64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
  di        27,390 pasangan =  3.3% dari total, di reducer R0
  corona    17,239 pasangan =  2.1% dari total, di reducer R2
  ini       10,124 pasangan =  1.2% dari total, di reducer R0
  yang       8,668 pasangan =  1.0% dari total, di reducer R3
  dan        6,785 pasangan =  0.8% dari total, di reducer R3


## H-12. Hadoop Streaming

### H-12.1. Menulis masukan dan skrip

Tulis `masukan.tsv`, `mapper.py`, serta `reducer.py`. Mapper membaca stdin dan mencetak `kata<TAB>1`. Reducer menjumlahkan key berurutan.


In [20]:
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8")

Path("mapper.py").write_text('''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:

            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')
print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))


Baris masukan: 90818


### H-12.2. Menjalankan pipe Linux

`sort` bertindak sebagai shuffle. `LC_ALL=C` mengurutkan key berdasarkan byte. Perintah berikut berjalan di Colab dan lingkungan shell Unix.


In [21]:
!head -n 3 masukan.tsv | python3 mapper.py

!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5


kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1


( cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > )  0.53s user 0.04s system 122% cpu 0.464 total


   32275 hasil_pipe.txt


di	27390
corona	17239
ini	10124
yang	8668
dan	6785
sort: Broken pipe


### H-12.3. Memastikan hasil sama dengan H-9

Baca hasil pipe dan bandingkan seluruh kamus, bukan hanya lima kata teratas.


In [22]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)
print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)


Hasil pipe sama dengan hasil H-9: True


### H-12. Opsional: menjalankan Hadoop asli

Jika waktu tersisa, Anda dapat menjalankan mapper dan reducer yang sama pada Hadoop sungguhan dalam local mode (satu mesin, tanpa HDFS dan YARN). Perintah berikut ditulis berdasarkan dokumentasi Hadoop 3.3.6. Bagian ini opsional dan belum dijalankan; jalankan sel ini hanya jika mengerjakan bagian opsional di Colab.

In [ ]:
%%bash
sudo apt-get install -y openjdk-11-jdk-headless
wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
tar -xzf hadoop-3.3.6.tar.gz
export HADOOP_HOME=$PWD/hadoop-3.3.6
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64


rm -rf keluaran_hadoop
$HADOOP_HOME/bin/hadoop jar \
  $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input masukan.tsv -output keluaran_hadoop

ls keluaran_hadoop
diff <(LC_ALL=C sort hasil_pipe.txt) keluaran_hadoop/part-00000 && echo "SAMA"


## H-13. Kata Teratas per Kategori, Ekspor ala Sqoop, dan Rangkuman

Ubah key menjadi `(kategori, kata)`, buang stopword di mapper, dan lakukan combiner lokal. Ambil lima kata teratas per kategori. Simpan hasil ke Parquet dan ekspor ke SQLite sebagai simulasi arah `sqoop export`.


In [23]:
def mapper_kategori(path):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1
    return c

gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))
for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

import sqlite3
SIAP = Path("siap_pakai"); SIAP.mkdir(exist_ok=True)
baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)
print("Baris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""SELECT kata, jumlah FROM kata_per_kategori
                    WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3""").fetchall())
db.close()


finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)
Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


## K. Pemeriksaan Output

Cocokkan angka tetap dengan tabel acuan modul. Waktu eksekusi dan judul crawl terbaru tidak dipaksa sama dengan contoh dosen. Pemeriksaan ini juga memverifikasi bahwa hasil MapReduce, combiner, dan pipe identik.

**Catatan hasil aktual:** jumlah pasangan tanpa combiner adalah **825.601**, sama dengan acuan modul. Seluruh pemeriksaan pada tabel sesuai, termasuk kesetaraan hasil Counter, MapReduce, combiner, dan pipe.

In [24]:
pemeriksaan = [
    ("Baris CSV", len(df_sumber), 91017),
    ("Halaman portal", N_HALAMAN, 911),
    ("Event masuk", statistik["masuk"], 91017),
    ("Event ditulis", statistik["keluar"], 91017),
    ("File batch JSONL", statistik["file"], 19),
    ("URL artikel tidak valid", int((~aturan_kualitas(mentah)["url artikel valid"]).sum()), 5640),
    ("Baris bersih", len(bersih), 90818),
    ("Input split", len(splits), 9),
    ("Pasangan tanpa combiner", total_pasangan, 825601),
    ("Kata unik", len(hasil_semua), 32275),
    ("Pasangan dengan combiner", sum(len(x) for x in h2), 74009),
    ("Hasil combiner = H-9", dict(total) == hasil_semua, True),
    ("Hasil pipe = H-9", hasil_pipe == hasil_semua, True),
]
tabel_cek = pd.DataFrame(pemeriksaan, columns=["pemeriksaan", "aktual", "acuan"])
tabel_cek["sesuai"] = tabel_cek["aktual"] == tabel_cek["acuan"]
print(tabel_cek.to_string(index=False))
beda = tabel_cek[~tabel_cek["sesuai"]]
if beda.empty:
    print("\nSeluruh angka tetap sesuai acuan modul.")
else:
    print("\nPerbedaan terhadap angka acuan dicatat, tidak diubah agar cocok.")
    print(beda.to_string(index=False))
assert dict(total) == hasil_semua == hasil_pipe
assert total_pasangan == sum(hasil_semua.values())
assert statistik["masuk"] == statistik["keluar"] == len(mentah)


             pemeriksaan aktual  acuan  sesuai
               Baris CSV  91017  91017    True
          Halaman portal    911    911    True
             Event masuk  91017  91017    True
           Event ditulis  91017  91017    True
        File batch JSONL     19     19    True
 URL artikel tidak valid   5640   5640    True
            Baris bersih  90818  90818    True
             Input split      9      9    True
 Pasangan tanpa combiner 825601 825601    True
               Kata unik  32275  32275    True
Pasangan dengan combiner  74009  74009    True
    Hasil combiner = H-9   True   True    True
        Hasil pipe = H-9   True   True    True

Seluruh angka tetap sesuai acuan modul.


## L. Analisis Hasil

### 1. Mengapa 2 proses tanpa combiner bisa lebih lambat daripada 1 proses? Apa yang berubah ketika combiner dipasang?

**Jawaban:** pekerjaan tokenisasi dataset ini relatif kecil. Menjalankan proses tambahan memerlukan pembuatan worker, pembacaan split, serialisasi, dan pemindahan hasil antarproses. Tanpa combiner, hasil pengujian menghasilkan 825.601 pasangan yang dikembalikan dari mapper, sesuai acuan modul. Biaya koordinasi dapat melebihi penghematan komputasi, sehingga 2 proses bisa lebih lambat.

Dengan combiner, mapper menjumlahkan kata secara lokal dan hanya mengirim 74.009 pasangan: berkurang **91,04%**, atau sekitar **11,16 kali** lebih sedikit. Hasil akhir tetap identik. Pengurangan pasangan tidak menjamin waktu 2 proses pasti lebih cepat; bandingkan median waktu yang muncul ketika H-10 dijalankan di Colab. Tabel di bawah memakai hasil eksekusi terbaru.


**Perbandingan waktu hasil eksekusi terbaru:**

| Skenario | Median waktu (detik) | Pasangan dikirim |
|---|---:|---:|
| 1 proses, tanpa combiner | 0,116 | 825.601 |
| 1 proses, dengan combiner | 0,137 | 74.009 |
| 2 proses, tanpa combiner | 0,207 | 825.601 |
| 2 proses, dengan combiner | 0,118 | 74.009 |

Angka ini berasal dari eksekusi terbaru pada mesin lokal dengan metode proses `fork`, seperti default Linux/Colab. Kode notebook tetap mengikuti modul. Waktu di Colab dapat berbeda.
Pada pengujian tanpa combiner, 2 proses lebih lambat daripada 1 proses (0,207 vs 0,116 detik).
Pada pengujian dengan combiner, 2 proses lebih cepat daripada 1 proses (0,118 vs 0,137 detik).

### 2. Mengapa menambah reducer dari 4 menjadi 64 tidak membuat beban reducer terberat mendekati bagian yang adil? Bagaimana combiner mengubahnya?

**Jawaban:** semua pasangan dengan key sama wajib masuk ke reducer yang sama. Kata `di` menghasilkan 27.390 pasangan, sekitar 3,32% dari seluruh pasangan. Pada 64 reducer, bagian adil hanya 1,56%, sehingga satu kata itu sendiri sudah melebihi bagian adil. Reducer terberat menerima 4,28%, sekitar 2,74 kali bagian adil; pada 4 reducer beban terberat 29,17% dibanding 25%.

Combiner mengirim satu pasangan per kata unik per mapper, sehingga kata sangat sering tidak lagi menghasilkan satu kiriman per kemunculan. Pada 4 reducer, pembagian pasangan menjadi sekitar 24,7%–25,6%. Combiner menekan skew lalu lintas pasangan; ia tidak otomatis menyamakan ukuran input split atau waktu kerja semua mapper.

### 3. Apakah `rp`, `ri`, dan `the` sebaiknya ditambahkan ke stopword? Jelaskan dari sisi tujuan analisis.

**Jawaban:** tergantung tujuan. Untuk merangkum topik umum, `rp` dan `ri` dapat menutupi kata yang lebih informatif, sedangkan `the` biasanya merupakan kata fungsi bahasa Inggris. Ketiganya masuk akal diuji sebagai stopword domain. Namun, pada analisis nilai uang, ekonomi, atau penyebutan negara, `rp` dan `ri` dapat berguna. Bandingkan top kata sebelum/sesudah dan catat alasannya; jangan membuang kata hanya karena frekuensinya tinggi.

### 4. Seluruh baris inet memiliki URL rusak. Apa dampaknya bila URL dipakai sebagai kunci join? Mengapa baris itu tidak dibuang?

**Jawaban:** 5.640 URL mentah inet berisi `#`. Bila dipakai sebagai kunci, URL tersebut tidak mengidentifikasi artikel secara unik; join dengan nilai `#` yang berulang dapat menghasilkan pasangan many-to-many dan menggandakan baris. Setelah URL dikosongkan, perilaku null juga harus diperhatikan: SQL biasanya tidak mencocokkan null, sedangkan pandas merge dapat mencocokkan kunci null antartabel. Karena itu keluarkan kunci tidak valid dari join atau gunakan ID pengganti yang tercatat.

Judulnya tetap bermanfaat untuk word count. Membuang seluruh baris inet menghilangkan satu kategori dan menimbulkan bias. Pipeline mempertahankan teksnya, mengosongkan URL/ID yang rusak, lalu tetap menerapkan deduplikasi serta batas panjang judul. Jumlah inet setelah cleaning menjadi 5.577.

### 5. Mengapa dipilih partisi kategori dibanding kategori + bulan untuk H-9? Hubungkan dengan jumlah mapper dan small file problem.

**Jawaban:** kategori menghasilkan 9 berkas, sehingga terdapat 9 tugas mapper, sesuai analisis per kategori. Kategori + bulan menghasilkan 54 berkas dengan total data hampir sama. Akibatnya ada lebih banyak pekerjaan kecil, pembukaan file, metadata, dan biaya penjadwalan. Pada dataset 90.818 judul, tambahan overhead tersebut dapat lebih besar daripada manfaat paralelismenya. Partisi kategori juga memungkinkan query finance membaca 1 dari 9 berkas. Pilihan ini tetap memiliki kelemahan: ukuran split news jauh lebih besar daripada sport.

### 6. Mengapa hasil harus diurutkan sebelum masuk ke reducer.py? Apa yang terjadi bila sort dihilangkan?

**Jawaban:** reducer hanya menyimpan total untuk key yang sedang dibaca. Ia mencetak hasil ketika key berubah, sehingga semua kemunculan key yang sama harus berurutan. Tanpa sort, kata yang muncul lagi setelah kata lain akan dicetak sebagai kelompok baru. Misalnya `a, b, a` menghasilkan dua baris untuk `a`, bukan satu total global. Mengubah keluaran seperti itu langsung menjadi dict juga dapat menimpa subtotal sebelumnya. `sort` menjalankan fungsi pengelompokan shuffle agar hasil pipe sama dengan H-9.

### 7. Mengapa H-4 memakai sitemap berita, bukan membuka setiap artikel? Apa untung ruginya bagi server dan kode?

**Jawaban:** sitemap berita sudah menyediakan URL, judul, tanggal, dan kadang kata kunci. Beberapa request XML cukup untuk mengambil 20 judul, sehingga lebih ringan bagi server daripada membuka 20 halaman artikel lengkap. Bagi kode, parser tidak bergantung pada layout HTML artikel yang mudah berubah.

Keterbatasannya: sitemap tidak menyediakan isi artikel lengkap, kata kunci dapat kosong, cakupannya mengikuti entri yang masih dimuat, dan kualitas metadata tanggal perlu diperiksa. Format sitemap atau aturan akses juga dapat berubah. Robots, batas request, jeda 2 detik, serta penghentian saat 403/429 tetap berlaku.

## M. Studi Kasus

**Soal:** pimpinan redaksi meminta ringkasan 150–200 kata berdasarkan H-7 dan H-13 yang memuat kapan topik corona memuncak, kategori yang paling banyak membahasnya dan kategori yang paling berbeda, serta satu keterbatasan data.

**Jawaban:**

Analisis 90.818 judul berita detik.com periode 1 Januari–13 Juni 2020 menunjukkan bahwa perhatian terhadap corona memuncak pada Maret. Jumlah judul yang memuat “corona” naik dari 385 pada Januari menjadi 1.077 pada Februari, lalu mencapai 6.994 pada Maret. Setelah itu jumlahnya turun menjadi 4.913 pada April dan 2.934 pada Mei. Juni mencatat 943 judul, tetapi hanya mencakup 13 hari sehingga tidak dapat dibandingkan langsung dengan bulan penuh.

Berdasarkan frekuensi token per kategori, news mencatat kemunculan “corona” terbanyak, yaitu 7.539, disusul health sebanyak 2.590 dan finance sebanyak 2.470. Angka ini menghitung kemunculan kata, bukan otomatis jumlah artikel unik, dan dipengaruhi banyaknya judul dalam tiap kategori. Kategori food memiliki pola lima kata teratas yang paling berbeda secara kualitatif: makanan, makan, foto, galeri, dan enak. Berbeda dari kategori lain, corona tidak masuk lima kata teratasnya.

Temuan tersebut menunjukkan dominasi tema pandemi sekaligus bertahannya topik kuliner. Namun, data hanya berisi judul, bukan isi artikel, sehingga tidak cukup untuk menyimpulkan kedalaman pembahasan atau sentimen. Ringkasan ini menggambarkan dataset yang tersedia, bukan seluruh pemberitaan sepanjang 2020.


## N. Latihan

### Latihan 1. Stopword domain

**Soal:** tambahkan 5 kata ke STOPWORD yang dianggap mengganggu hasil, jalankan ulang H-13, dan catat perubahan lima kata teratas setiap kategori.

**Jawaban dan alasan:** tambahan yang diuji adalah `rp`, `ri`, `the`, `foto`, dan `galeri`. Dua kata pertama merupakan singkatan umum, `the` kata fungsi Inggris, sedangkan `foto` dan `galeri` lebih sering menandai format penyajian. Pilihan ini ditujukan untuk ringkasan topik; kata-kata tersebut belum tentu harus dibuang untuk tujuan lain. STOPWORD dasar dan hasil H-13 dipertahankan agar perbandingan dapat diulang.


In [25]:
TAMBAHAN_STOPWORD = {"rp", "ri", "the", "foto", "galeri"}
STOPWORD_DOMAIN = STOPWORD | TAMBAHAN_STOPWORD

def mapper_kategori_domain(path):
    kategori = path.parent.name.split("=", 1)[1]
    hitungan = collections.Counter()
    for judul in pd.read_parquet(path, columns=["judul"])["judul"]:
        for kata in tokenisasi(judul):
            if kata not in STOPWORD_DOMAIN:
                hitungan[(kategori, kata)] += 1
    return hitungan

hasil_domain = collections.Counter()
for hasil_lokal in map(mapper_kategori_domain, splits):
    hasil_domain.update(hasil_lokal)

def top5_kategori(hitungan, kategori):
    return sorted([(n, k) for (kat, k), n in hitungan.items() if kat == kategori], reverse=True)[:5]

perbandingan_stopword = []
for kategori in sorted(KATEGORI_DIKENAL):
    sebelum = top5_kategori(gabung, kategori)
    sesudah = top5_kategori(hasil_domain, kategori)
    lama = {k for n, k in sebelum}
    baru = {k for n, k in sesudah}
    perbandingan_stopword.append({
        "kategori": kategori,
        "sebelum": ", ".join(f"{k} ({n})" for n, k in sebelum),
        "sesudah": ", ".join(f"{k} ({n})" for n, k in sesudah),
        "keluar": ", ".join(sorted(lama - baru)) or "—",
        "masuk": ", ".join(sorted(baru - lama)) or "—",
    })
tabel_stopword = pd.DataFrame(perbandingan_stopword)
print("Tambahan:", sorted(TAMBAHAN_STOPWORD))
print(tabel_stopword.to_string(index=False))
pd.DataFrame([(kat, kata, n) for (kat, kata), n in hasil_domain.items()],
             columns=["kategori", "kata", "jumlah"]).to_parquet(SIAP / "latihan1_stopword_domain.parquet", index=False)


Tambahan: ['foto', 'galeri', 'ri', 'rp', 'the']
kategori                                                                     sebelum                                                                     sesudah       keluar        masuk
 finance              corona (2470), rp (1385), ri (1125), harga (701), jokowi (642)          corona (2470), harga (701), jokowi (642), mau (562), ekonomi (536)       ri, rp ekonomi, mau
    food            makanan (699), makan (539), foto (458), galeri (438), enak (358)            makanan (699), makan (539), enak (358), bikin (343), rumah (311) foto, galeri bikin, rumah
  health corona (2590), virus (1040), pasien (471), indonesia (427), meninggal (413) corona (2590), virus (1040), pasien (471), indonesia (427), meninggal (413)            —            —
     hot          corona (1658), anak (793), rumah (561), meninggal (545), the (528)         corona (1658), anak (793), rumah (561), meninggal (545), film (489)          the         film
    inet       co

**Hasil perubahan:** finance mengeluarkan ri/rp dan memasukkan ekonomi/mau; food mengeluarkan foto/galeri dan memasukkan bikin/rumah; hot mengganti the dengan film; oto mengganti rp dengan honda; travel mengganti foto dengan indonesia. Top-5 health, inet, news, dan sport tetap. Tabel output di atas mencatat frekuensi sebelum dan sesudah untuk seluruh kategori.


### Latihan 2. Per bulan

**Soal:** ubah key H-13 menjadi `(bulan, kata)`. Kata apa yang berubah dari Januari ke Mei?

**Pengerjaan:** pakai stopword dasar agar perubahan bulanan tidak tercampur dengan perubahan daftar stopword Latihan 1. Mapper menghitung key gabungan bulan–kata pada tiap split, combiner menghitung lokal, dan reducer menjumlahkan seluruh split. Tampilkan lima kata teratas setiap bulan serta perbandingan Januari dan Mei.


**Hasil:** top-5 Januari adalah banjir (1.220), rp (616), polisi (554), jokowi (516), dan tahun (493). Pada Mei berubah menjadi corona (2.935), kasus (640), positif (628), pandemi (596), dan warga (565). Seluruh anggota top-5 berganti, dari topik banjir dan berita umum menuju pandemi. Ini perbandingan frekuensi token; angka judul yang memuat corona pada Mei adalah 2.934.


In [26]:
def mapper_bulan(path):
    hitungan = collections.Counter()
    frame = pd.read_parquet(path, columns=["bulan", "judul"])
    for bulan, judul in frame.itertuples(index=False, name=None):
        for kata in tokenisasi(judul):
            if kata not in STOPWORD:
                hitungan[(bulan, kata)] += 1
    return hitungan

hasil_bulan = collections.Counter()
for hitungan in map(mapper_bulan, splits):
    hasil_bulan.update(hitungan)

puncak_bulanan = {}
for bulan in sorted(bersih["bulan"].unique()):
    puncak_bulanan[bulan] = sorted([(n, k) for (b, k), n in hasil_bulan.items() if b == bulan], reverse=True)[:5]
    print(bulan, ", ".join(f"{k} ({n:,})" for n, k in puncak_bulanan[bulan]))

jan = {k for n, k in puncak_bulanan["2020-01"]}
mei = {k for n, k in puncak_bulanan["2020-05"]}
print("\nKeluar dari top-5 Januari:", sorted(jan - mei))
print("Masuk top-5 Mei:", sorted(mei - jan))
print("Tetap di top-5:", sorted(jan & mei))
print("\nFrekuensi token corona dan proporsi judul:")
for bulan in ("2020-01", "2020-05"):
    bagian = bersih[bersih["bulan"] == bulan]
    judul_corona = bagian["judul"].str.lower().str.contains("corona", regex=False).sum()
    print(f"{bulan}: token corona={hasil_bulan[(bulan, 'corona')]:,}; "
          f"judul memuat corona={judul_corona:,}/{len(bagian):,} ({judul_corona/len(bagian)*100:.2f}%)")


2020-01 banjir (1,220), rp (616), polisi (554), jokowi (516), tahun (493)
2020-02 corona (1,076), polisi (630), virus (596), rp (572), banjir (572)
2020-03 corona (6,986), virus (1,340), positif (925), pasien (749), rumah (695)
2020-04 corona (4,914), positif (828), pandemi (706), psbb (689), warga (609)
2020-05 corona (2,935), kasus (640), positif (628), pandemi (596), warga (565)
2020-06 corona (943), normal (471), new (465), kasus (315), rp (265)

Keluar dari top-5 Januari: ['banjir', 'jokowi', 'polisi', 'rp', 'tahun']
Masuk top-5 Mei: ['corona', 'kasus', 'pandemi', 'positif', 'warga']
Tetap di top-5: []

Frekuensi token corona dan proporsi judul:
2020-01: token corona=385; judul memuat corona=385/16,258 (2.37%)
2020-05: token corona=2,935; judul memuat corona=2,934/16,352 (17.94%)


**Pembahasan:** seluruh anggota top-5 Januari keluar dan digantikan lima kata bertema pandemi pada Mei. Peringkat menggambarkan pergeseran kosakata judul. Jumlah judul yang memuat corona meningkat dari 385 pada Januari menjadi 2.934 pada Mei. Frekuensi token dan jumlah judul harus dibedakan karena satu judul dapat memuat token yang sama lebih dari sekali; proporsi terhadap semua judul per bulan membantu perbandingan ketika volume berita berbeda.

### Latihan 3. Bigram

**Soal:** ubah mapper agar menghasilkan pasangan kata berurutan, misalnya `virus corona`. Berapa pasangan yang dihasilkan dan berapa setelah combiner?

**Pengerjaan:** bigram dibuat dari token yang bersebelahan **dalam satu judul**, tanpa melompati batas judul. Stopword tidak dibuang agar hasil mengikuti ketetanggaan token asli, seperti H-9 yang menghitung semua kata.


In [27]:
def mapper_bigram(path):
    pasangan = []
    for judul in pd.read_parquet(path, columns=["judul"])["judul"]:
        kata = tokenisasi(judul)
        pasangan.extend((f"{a} {b}", 1) for a, b in zip(kata, kata[1:]))
    return pasangan

def mapper_bigram_combiner(path):
    hitungan = collections.Counter()
    for judul in pd.read_parquet(path, columns=["judul"])["judul"]:
        kata = tokenisasi(judul)
        hitungan.update(f"{a} {b}" for a, b in zip(kata, kata[1:]))
    return hitungan

bigram_tanpa = [mapper_bigram(p) for p in splits]
bigram_dengan = [mapper_bigram_combiner(p) for p in splits]
jumlah_bigram = sum(len(h) for h in bigram_tanpa)
jumlah_bigram_combiner = sum(len(h) for h in bigram_dengan)
reducer_bigram = collections.Counter()
for hasil_lokal in bigram_dengan:
    reducer_bigram.update(hasil_lokal)
cek_bigram = collections.Counter()
for pasangan in bigram_tanpa:
    for k, n in pasangan:
        cek_bigram[k] += n
assert cek_bigram == reducer_bigram
rumus_bigram = sum(max(len(tokenisasi(j)) - 1, 0) for j in bersih["judul"])
assert jumlah_bigram == rumus_bigram
print(f"Pasangan tanpa combiner : {jumlah_bigram:,}")
print(f"Pasangan dengan combiner: {jumlah_bigram_combiner:,}")
print(f"Bigram unik global      : {len(reducer_bigram):,}")
print(f"Pengurangan transfer    : {(1-jumlah_bigram_combiner/jumlah_bigram)*100:.2f}%")
print("Hasil akhir sama        :", cek_bigram == reducer_bigram)
print("\n10 bigram teratas:")
for kata, n in reducer_bigram.most_common(10):
    print(f"{kata:<30} {n:>8,}")


Pasangan tanpa combiner : 734,783
Pasangan dengan combiner: 411,217
Bigram unik global      : 349,114
Pengurangan transfer    : 44.04%
Hasil akhir sama        : True

10 bigram teratas:
virus corona                      2,915
corona di                         2,178
positif corona                    1,804
di tengah                         1,258
pandemi corona                    1,075
di rumah                          1,047
new normal                          917
di indonesia                        871
rp juta                             700
hari ini                            627


**Jawaban hasil eksekusi:** terdapat **734.783 pasangan bigram** tanpa combiner dan **411.217 pasangan** setelah combiner, dengan **349.114 bigram unik global**. Pengurangan transfer sekitar **44,04%** dan hasil reduce kedua cara sama.

**Pembahasan:** satu judul dengan n token menghasilkan maksimum n−1 bigram. Jumlah pasangan setelah combiner adalah penjumlahan bigram unik per split, sehingga umumnya lebih besar daripada jumlah bigram unik global. Combiner mengurangi data yang dikirim, sementara frekuensi akhirnya tetap identik.

### Latihan 4. Jumlah reducer

**Soal:** jalankan H-11 dengan R = 2, 8, dan 32. Buat tabel beban reducer terberat dibanding bagian yang adil.

**Pengerjaan:** pertahankan CRC32 dan input mapper yang sama. Selain skenario tanpa combiner yang diminta, tabel menyertakan pembanding dengan combiner. Rasio > 1 berarti reducer terberat mendapat beban di atas bagian adil.


In [28]:
baris_reducer = []
for r in (2, 8, 32):
    b_tanpa = beban(hasil_map, r)
    b_dengan = beban([list(c.items()) for c in h2], r)
    adil = 100 / r
    terberat = max(b_tanpa) / sum(b_tanpa) * 100
    baris_reducer.append({
        "R": r,
        "pasangan reducer terberat": max(b_tanpa),
        "terberat tanpa (%)": terberat,
        "bagian adil (%)": adil,
        "rasio terhadap adil": terberat / adil,
        "terberat combiner (%)": max(b_dengan) / sum(b_dengan) * 100,
    })
tabel_reducer = pd.DataFrame(baris_reducer)
print(tabel_reducer.to_string(index=False, float_format=lambda x: f"{x:.3f}"))


 R  pasangan reducer terberat  terberat tanpa (%)  bagian adil (%)  rasio terhadap adil  terberat combiner (%)
 2                     431563              52.273           50.000                1.045                 50.234
 8                     138978              16.834           12.500                1.347                 12.852
32                      48073               5.823            3.125                1.863                  3.439


**Jawaban hasil eksekusi:**

| R | Pasangan reducer terberat | Beban terberat | Bagian adil | Rasio terhadap adil |
|---|---:|---:|---:|---:|
| 2 | 431.563 | 52,273% | 50,000% | 1,045 |
| 8 | 138.978 | 16,834% | 12,500% | 1,347 |
| 32 | 48.073 | 5,823% | 3,125% | 1,863 |

**Pembahasan:** bandingkan rasio terhadap bagian adil, bukan persentase terberat saja. Saat jumlah reducer naik, persentase terberat bisa turun tetapi ketimpangan relatif meningkat karena satu key tetap tidak dapat dibagi ke banyak reducer. Tabel menghitung beban pasangan transfer; ia bukan pengukuran waktu reducer pada cluster.

### Latihan 5. Crawl ulang

**Soal:** tambahkan jeda 0,05 detik pada `crawl_iter` untuk 50 halaman pertama. Berapa waktunya, dan berapa lama bila 911 halaman memakai jeda 1 detik?

**Pengerjaan:** pengujian hanya dilakukan pada portal lokal. `crawl_iter` versi modul tidur sekali per halaman, termasuk setelah halaman terakhir, sehingga tambahan jeda untuk 50 halaman adalah 2,5 detik dan untuk 911 halaman adalah 911 detik.


In [29]:
t0_latihan5 = time.perf_counter()
hasil_50 = list(crawl_iter(BASE, range(1, 51), delay=0.05))
waktu_50 = time.perf_counter() - t0_latihan5
waktu_nonjeda_50 = max(0.0, waktu_50 - 50 * 0.05)
estimasi_nonjeda_911 = waktu_nonjeda_50 / 50 * N_HALAMAN
estimasi_911 = N_HALAMAN * 1.0 + estimasi_nonjeda_911
print("Halaman diuji          :", 50)
print("Judul terkumpul        :", len(hasil_50))
print(f"Waktu aktual 50 halaman : {waktu_50:.3f} detik")
print(f"Jeda minimum 50 halaman : {50*0.05:.1f} detik")
print(f"Jeda saja 911 halaman   : {N_HALAMAN:.0f} detik = {N_HALAMAN/60:.2f} menit")
print(f"Estimasi total 911     : {estimasi_911:.2f} detik = {estimasi_911/60:.2f} menit")
assert len(hasil_50) == 5000


Halaman diuji          : 50
Judul terkumpul        : 5000
Waktu aktual 50 halaman : 3.480 detik
Jeda minimum 50 halaman : 2.5 detik
Jeda saja 911 halaman   : 911 detik = 15.18 menit
Estimasi total 911     : 928.86 detik = 15.48 menit


**Jawaban hasil eksekusi:** 50 halaman (5.000 judul) dengan jeda 0,05 detik memerlukan **3,480 detik**. Estimasi 911 halaman dengan jeda 1 detik adalah **928,86 detik**, sekitar **15 menit 28,9 detik**, termasuk overhead yang diekstrapolasi dari pengujian 50 halaman.

**Pembahasan:** 911 detik sama dengan 15 menit 11 detik untuk jeda saja. Waktu total lebih lama karena request lokal, pembuatan HTML, parsing, dan kemungkinan retry. Estimasi di atas mengekstrapolasi overhead dari 50 halaman; pengujian 911 halaman dengan jeda 1 detik tidak dijalankan, sehingga angka estimasi bukan waktu aktual.

### Latihan 6. Sitemap lain

**Soal:** ganti satu alamat di `SITEMAP_DIPILIH` dengan `sitemap_news` dari bagian lain yang tercantum di `sitemap.xml`. Kategori apa yang muncul? Terapkan `aturan_kualitas` H-6 pada 20 judul hasil crawl dan laporkan hasilnya.

**Pengerjaan:** baca indeks sitemap, pilih satu sitemap berita lain, lalu ganti alamat ketiga. Jika crawl H-4 berhasil, gunakan cache untuk dua sitemap yang tetap dan hanya request sitemap pengganti; jumlah akhir tetap maksimal 7 + 7 + 6 = 20. Jika akses ditolak, catat kegagalan tanpa mengarang judul atau hasil kualitas. Semua request memakai fungsi pengambilan H-4 beserta robots dan jedanya.


In [30]:
sampel_latihan6 = []
status_latihan6 = {}
SITEMAP_LATIHAN6 = list(SITEMAP_DIPILIH)
try:
    sesi_latihan6 = requests.Session()
    sesi_latihan6.headers["User-Agent"] = UA_DETIK
    INDEX_SITEMAP = "https://www.detik.com/sitemap.xml"
    isi_indeks = ambil_xml(INDEX_SITEMAP, sesi_latihan6)
    sup_indeks = BeautifulSoup(isi_indeks, "xml")
    alamat_indeks = [x.get_text(strip=True) for x in sup_indeks.find_all("loc")]
    calon = [u for u in alamat_indeks
             if urlparse(u).path.endswith("sitemap_news.xml")
             and u not in SITEMAP_DIPILIH
             and urlparse(u).hostname
             and urlparse(u).hostname.endswith(".detik.com")
             and urlparse(u).hostname.split(".")[0] in KATEGORI_DIKENAL]
    if not calon:
        raise ValueError("Tidak ada sitemap berita lain yang sesuai pada indeks saat ini.")
    # Prioritaskan kategori yang belum dipakai di H-4.
    kategori_awal = {urlparse(u).netloc.split(".")[0] for u in SITEMAP_DIPILIH}
    calon.sort(key=lambda u: (urlparse(u).netloc.split(".")[0] in kategori_awal, u))
    sitemap_baru = calon[0]
    SITEMAP_LATIHAN6[2] = sitemap_baru
    print("Sitemap awal :", SITEMAP_DIPILIH)
    print("Sitemap baru :", SITEMAP_LATIHAN6)
    for posisi, alamat in enumerate(SITEMAP_LATIHAN6):
        kategori_url = urlparse(alamat).netloc.split(".")[0]
        cache = [x for x in sampel if x["kategori"] == kategori_url] if posisi < 2 else []
        entri = cache[:BATAS_PER_BAGIAN] if len(cache) >= BATAS_PER_BAGIAN else parse_sitemap_news(ambil_xml(alamat, sesi_latihan6))[:BATAS_PER_BAGIAN]
        sisa = BATAS_TOTAL - len(sampel_latihan6)
        sampel_latihan6.extend(entri[:sisa])
        if len(sampel_latihan6) >= BATAS_TOTAL:
            break
    if not sampel_latihan6:
        raise ValueError("Sitemap tidak mengandung entri berita yang dapat diparsing.")
    frame_latihan6 = pd.DataFrame(sampel_latihan6)
    print("\nJumlah judul:", len(frame_latihan6))
    print("Kategori:", frame_latihan6["kategori"].value_counts().to_dict())
    kualitas_latihan6 = pd.DataFrame([
        {"aturan": nama, "lulus": int(lulus.sum()), "gagal": int((~lulus).sum())}
        for nama, lulus in aturan_kualitas(frame_latihan6).items()
    ])
    print(kualitas_latihan6.to_string(index=False))
    if len(frame_latihan6) != 20:
        print("Catatan: entri tersedia kurang dari target 20; jumlah aktual dilaporkan apa adanya.")
    Path("contoh_crawl").mkdir(exist_ok=True)
    frame_latihan6.to_json("contoh_crawl/latihan6_sitemap_lain.jsonl", orient="records", lines=True, force_ascii=False)
    status_latihan6 = {"berhasil": True, "jumlah": len(frame_latihan6), "sitemap_baru": sitemap_baru,
                       "kategori": frame_latihan6["kategori"].value_counts().to_dict()}
except Exception as e:
    sampel_latihan6 = []  # Jangan memakai hasil parsial sebagai hasil target lengkap.
    status_latihan6 = {"berhasil": False, "error": f"{type(e).__name__}: {e}"}
    print("Latihan 6 belum menghasilkan sampel:", status_latihan6["error"])
print("Status aktual:", status_latihan6)


Sitemap awal : ['https://news.detik.com/berita/sitemap_news.xml', 'https://inet.detik.com/news/sitemap_news.xml', 'https://sport.detik.com/basket/sitemap_news.xml']
Sitemap baru : ['https://news.detik.com/berita/sitemap_news.xml', 'https://inet.detik.com/news/sitemap_news.xml', 'https://finance.detik.com/berita-ekonomi-bisnis/sitemap_news.xml']



Jumlah judul: 20
Kategori: {'news': 7, 'inet': 7, 'finance': 6}
              aturan  lulus  gagal
  judul tidak kosong     20      0
     judul 3-30 kata     20      0
    kategori dikenal     20      0
       tanggal valid     20      0
   url artikel valid     20      0
judul tidak duplikat     20      0
Status aktual: {'berhasil': True, 'jumlah': 20, 'sitemap_baru': 'https://finance.detik.com/berita-ekonomi-bisnis/sitemap_news.xml', 'kategori': {'news': 7, 'inet': 7, 'finance': 6}}


**Jawaban hasil eksekusi:** sitemap basket diganti dengan `https://finance.detik.com/berita-ekonomi-bisnis/sitemap_news.xml`, yang ditemukan dalam indeks. Terkumpul **20 judul: news 7, inet 7, finance 6**. Hasil aturan kualitas: judul tidak kosong: 20 lulus, 0 gagal; judul 3-30 kata: 20 lulus, 0 gagal; kategori dikenal: 20 lulus, 0 gagal; tanggal valid: 20 lulus, 0 gagal; url artikel valid: 20 lulus, 0 gagal; judul tidak duplikat: 20 lulus, 0 gagal.

**Pembahasan:** kategori dan kegagalan aturan kualitas dilaporkan dari data aktual pada output sel. Sitemap baru belum tentu menghasilkan kategori baru: beberapa bagian bisa memakai subdomain kategori yang sama. Aturan 3–30 kata berasal dari dataset historis; judul terbaru yang melanggarnya harus dicatat, bukan dipaksa lolos. Jika akses gagal, bagian empiris ini memerlukan eksekusi ulang saat situs mengizinkan akses.

### Latihan 7. Kata kunci

**Soal:** kolom `kata_kunci` berisi kata kunci tiap berita yang dipisahkan koma. Tulis mapper dan reducer untuk menghitung kata kunci terbanyak pada hasil crawl.

**Pengerjaan:** satu kata kunci yang mengandung spasi tetap dianggap satu key, misalnya `mahkamah konstitusi`. Normalisasi huruf dan spasi, lalu abaikan entri kosong. Gunakan sampel H-4; jika tidak tersedia, gunakan crawl Latihan 6 yang berhasil. Contoh XML pengujian tidak dijadikan data crawl.


In [31]:
def mapper_kata_kunci(entri):
    for berita in entri:
        isi = berita.get("kata_kunci") or ""
        for kunci in isi.split(","):
            kunci = re.sub(r"\s+", " ", kunci.strip().lower())
            if kunci:
                yield kunci, 1

def reducer_kata_kunci(kunci, nilai):
    return kunci, sum(nilai)

sumber_kata_kunci = sampel if sampel else sampel_latihan6
hasil_kata_kunci = {}
if sumber_kata_kunci:
    shuffle_kunci = collections.defaultdict(list)
    for kunci, nilai in mapper_kata_kunci(sumber_kata_kunci):
        shuffle_kunci[kunci].append(nilai)
    hasil_kata_kunci = dict(reducer_kata_kunci(k, v) for k, v in shuffle_kunci.items())
    print("Sumber:", "H-4" if sampel else "Latihan 6")
    print("Jumlah judul:", len(sumber_kata_kunci))
    print("Judul tanpa kata kunci:", sum(not (b.get("kata_kunci") or "").strip() for b in sumber_kata_kunci))
    print("Total pasangan mapper:", sum(map(len, shuffle_kunci.values())))
    print("Kata kunci unik:", len(hasil_kata_kunci))
    for kunci, frekuensi in collections.Counter(hasil_kata_kunci).most_common(10):
        print(f"{kunci:<45} {frekuensi:>4}")
    if not hasil_kata_kunci:
        print("Tidak ada metadata kata kunci terisi; frekuensi tidak dapat ditentukan.")
else:
    print("Belum dapat menghitung kata kunci: tidak ada hasil crawl langsung yang berhasil.")
    print("Jalankan kembali H-4 atau Latihan 6 saat akses situs tersedia, lalu jalankan sel ini.")


Sumber: H-4
Jumlah judul: 20
Judul tanpa kata kunci: 0
Total pasangan mapper: 73
Kata kunci unik: 63
ibl                                              4
kedaulatan digital                               3
kecelakaan truk                                  2
bekasi                                           2
jabodetabek                                      2
ibl all indonesian 2026                          2
ibl all indonesian                               2
larangan vape                                    1
polri                                            1
narkotika                                        1


**Jawaban hasil eksekusi:** 20 judul crawl menghasilkan **73 pasangan mapper** dan **63 kata kunci unik**. Judul tanpa metadata kata kunci: **0**. Sepuluh kata kunci teratas: **ibl (4)**, **kedaulatan digital (3)**, **bekasi (2)**, **ibl all indonesian (2)**, **ibl all indonesian 2026 (2)**, **jabodetabek (2)**, **kecelakaan truk (2)**, **ai (1)**, **aksara jawa (1)**, **andromeda (1)**. Urutan antarkata dengan frekuensi sama pada ringkasan ini diurutkan secara alfabetis; frekuensinya sama dengan output kode.

**Pembahasan:** frekuensi di sini menghitung kemunculan label dalam metadata, bukan token penyusun judul. Frasa tidak dipecah menjadi kata tunggal. Metadata kosong tidak membuktikan bahwa suatu topik tidak dibahas; hasil hanya mewakili label yang tersedia pada sampel kecil saat crawl.
